# MCMC posteriors and ensemble policy (village 50)

Fits the spike-and-slab Bayesian group Ising model by MCMC (PyMC), plots the posteriors, and trains an ensemble of CQL agents from posterior draws (majority-vote decision tree).

**Reads** `data/villages/adjacency_all.json`. **Writes** figures in `figures/`; the saved draws and tree records come from `ensemble_v50/run_v50.py`, and the paper figures from `plotting/make_posterior_v50.py` and `plotting/make_ensemble_tree.py`.

Run with this folder as working directory, after `pip install -e .` from the repository root.

# MCMC posteriors and ensemble policy

MCMC fit for the selected village, posterior plots, then an ensemble of N_ENSEMBLE CQL agents from posterior draws.

In [ ]:
import os, sys, shutil, tempfile, glob as _glob, json, time as _time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as _mlines
import matplotlib.patches as _mpatches
warnings.filterwarnings("ignore")

MF_DIR = os.path.dirname(os.path.abspath("mcmc_ensemble.ipynb"))

import igraph as ig
import d3rlpy
from qising.baselines import (
    DegreeCentralityPolicy, LIRPolicy, DegreeBinPolicy, QIsingPolicy)
from qising.environments.edge_bet_env import EdgeBetCommunitySISEnv
from qising.ising.group_ising import fit_group_ising_emvs
from qising.utils import build_village_envdata, run_policy_episodes
import logging; logging.getLogger("d3rlpy").setLevel(logging.WARNING)

ADJ_JSON = os.path.join(MF_DIR, "..", "..", "data", "villages", "adjacency_all.json")
FIG_DIR  = os.path.join(MF_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=150, bbox_inches="tight")
    print(f"  Saved: {name}.pdf / .png")

#Select the village
DEV_VILLAGE = "50"

# ── Hyperparameters — identical to main experiment ──────────────────
BETA_LIST  = [0.01, 0.5, 0.05, 0.07, 0.06, 0.02, 0.01, 0.4, 0.1, 0.3]
DELTA_LIST = [0.5,  0.9, 0.9,  0.6,  0.5,  0.50, 0.7,  0.6, 0.5, 0.8]

COMM_METHOD   = "edge_betweenness"
MIN_COMM_SIZE = 10

T_TRAIN        = 500
N_TRAIN_EPS    = 1
DATA_SEED      = 42
T_TEST         = 25
N_TEST_EPS     = 50
SKIP_PROB      = 0
INIT_RATE      = 0.0
TEST_INIT_RATE = 0.0
K_SEEDS        = T_TEST

DISCOUNT          = 0.8
ALPHA_CQL         = 0.1


V0_E, V1_E, C_E, SIGMA_E = 0.01, 10.0, 1.0, 10.0

V0_MCMC       = 0.01
V1_MCMC       = 10.0
C_PRIOR       = 1.0
SIGMA_SQ_MCMC = 10.0
N_MCMC_DRAWS  = 200
N_MCMC_TUNE   = 300
N_MCMC_CHAINS = 1
MCMC_SEED     = 42
MAX_MCMC_ROWS = 8_000

N_ENSEMBLE    = 20
ENSEMBLE_SEED = 1234
EVAL_SEED     = 99

USE_BEHAVIORAL_SNAPSHOT = False

POLICY_DISPLAY = {
    "degree":           "Degree",
    "lir":              "LIR",
    "degree_bin":       "Degree-Bin",
    "q_ising":          "EdgeBet Q-Ising",
    "q_ising_ensemble": "Ensemble Q-Ising",
}
POLICY_COLORS = {
    "degree":           "#e67e22",
    "lir":              "#9b59b6",
    "degree_bin":       "#f39c12",
    "q_ising":          "#27ae60",
    "q_ising_ensemble": "#e74c3c",
}

print("Imports OK.", f"d3rlpy {d3rlpy.__version__}")

In [ ]:
# ── Load adjacency and build village 50 data ─────────────────────────────
adj_all = json.load(open(ADJ_JSON))
A   = np.array(adj_all[DEV_VILLAGE], dtype=float)
np.fill_diagonal(A, 0)

dvd = build_village_envdata(
    A, BETA_LIST, DELTA_LIST,
    comm_method=COMM_METHOD, min_comm_size=MIN_COMM_SIZE)
_G  = dvd["n_bins"]

print(f"Village {DEV_VILLAGE}: N={dvd['N']}, n_bins={_G}")
print(f"  Bin sizes: {dvd['bin_counts'].tolist()}")
print(f"  Beta  per bin: {[round(dvd['beta_vec'][dvd['block_labels']==b][0], 3) for b in range(_G)]}")
print(f"  Delta per bin: {[round(dvd['delta_vec'][dvd['block_labels']==b][0], 3) for b in range(_G)]}")

In [ ]:
# ── Collect behavioral trajectories for the village ───────────────────────
dev_trajs = []
print(f"Collecting {N_TRAIN_EPS} behavioral episode(s) (T={T_TRAIN}) ...")
_t0 = _time.perf_counter()
for _ep in range(N_TRAIN_EPS):
    _env_ep = EdgeBetCommunitySISEnv(
        dvd["adj"], dvd["beta_vec"], dvd["delta_vec"], dvd["block_labels"],
        T=T_TRAIN, seed=DATA_SEED + _ep, init_rate=INIT_RATE)
    _traj = _env_ep.collect_behavioral_episode(skip_prob=SKIP_PROB, verbose=True)
    dev_trajs.append(_traj)
print(f"Done ({_time.perf_counter()-_t0:.1f}s)")

# Build reference env
_dev_env_ref = EdgeBetCommunitySISEnv(
    dvd["adj"], dvd["beta_vec"], dvd["delta_vec"], dvd["block_labels"],
    T=T_TRAIN, seed=DATA_SEED, init_rate=INIT_RATE)
_nb = _G

# Prepare arrays for EMVS / MCMC
dev_X_arr       = np.stack([t["adopted"][:-1].astype(float) for t in dev_trajs])
dev_y_arr       = np.stack([t["adopted"][1:].astype(float)  for t in dev_trajs])
dev_treated_arr = np.stack([t["seeded_nodes"][:-1]          for t in dev_trajs])

print(f"Training arrays: X={dev_X_arr.shape}, y={dev_y_arr.shape}")

In [ ]:
# ── EMVS fit — MAP estimates used as reference lines in posterior plots ───
print(f"Fitting EMVS (G={_G}, V0={V0_E}, V1={V1_E}, c={C_E}) ...")
_t0 = _time.perf_counter()
dev_emvs_result = fit_group_ising_emvs(
    dev_X_arr, dev_y_arr, dev_treated_arr, dvd["adj"], dvd["block_labels"],
    v0=V0_E, v1=V1_E, c=C_E, sigma_sq=SIGMA_E, verbose=True)
print(f"Done ({_time.perf_counter()-_t0:.1f}s)")

ext_e = dev_emvs_result["extracted"]
print(f"  Intercept:        {ext_e['intercept_mean'].round(3)}")
print(f"  Persistence:      {ext_e['rho_mean'].round(3)}")
print(f"  Direct treatment: {ext_e['beta_treat_mean'].round(3)}")
print(f"  Spillover:        {ext_e['spillover_mean'].round(3)}")

## MCMC posterior inference

Spike-and-slab prior on the couplings (γ), normal prior on intercept, persistence, direct treatment and spillover.

In [ ]:
# ── Build per-group design matrices for MCMC ────────────────────────────
# Layout: [1, y_j, treat_j, spill, counts_{g_0}...counts_{g_{G-1}}]
import pymc as pm

n_base = 4  # intercept, persistence, direct treatment, spillover
n_feat = n_base + _G
g_idx_v = dev_emvs_result["g_idx"]
nl_v    = dev_emvs_result["neighbor_lists"]
ng_v    = dev_emvs_result["neigh_g_idx"]

X_mcmc = [[] for _ in range(_G)]
y_mcmc = [[] for _ in range(_G)]

for _traj in dev_trajs:
    _Xarr   = _traj["adopted"][:-1].astype(float)
    _yarr   = _traj["adopted"][1:].astype(float)
    _seeded = _traj["seeded_nodes"][:-1]
    T_ep, N_v = _Xarr.shape
    _ti = np.zeros((T_ep, N_v), dtype=bool)
    for t in range(T_ep):
        idx = int(_seeded[t])
        if 0 <= idx < N_v:
            _ti[t, idx] = True

    for j in range(N_v):
        gi    = int(g_idx_v[j])
        neigh = nl_v[j]; ng_j = ng_v[j]; d = len(neigh)
        for t in range(T_ep):
            row = np.zeros(n_feat)
            row[0] = 1.0
            row[1] = _Xarr[t, j]
            row[2] = float(_ti[t, j])
            if d > 0:
                row[3] = _ti[t, neigh].sum()
                ns = _Xarr[t, neigh]
                for gm in range(_G):
                    row[n_base + gm] = ns[ng_j == gm].sum()
            X_mcmc[gi].append(row)
            y_mcmc[gi].append(int(_yarr[t, j]))

X_mcmc = [np.array(x) for x in X_mcmc]
y_mcmc = [np.array(y) for y in y_mcmc]
print(f"Design matrices ready. Rows per group: {[len(y) for y in y_mcmc]}")

# ── Run per-group MCMC ───────────────────────────────────────────────────
pi_coup    = min(C_PRIOR / _G, 0.99)
mcmc_traces = {}
print(f"\nRunning MCMC (G={_G}, draws={N_MCMC_DRAWS}, tune={N_MCMC_TUNE}, chains={N_MCMC_CHAINS}) ...")

for gi in range(_G):
    Xg = X_mcmc[gi]; yg = y_mcmc[gi]
    if len(np.unique(yg)) < 2:
        print(f"  Group {gi}: single class — using EMVS MAP fallback")
        mcmc_traces[gi] = None
        continue
    if len(yg) > MAX_MCMC_ROWS:
        _rng_sub = np.random.default_rng(MCMC_SEED + gi + 100)
        _idx_sub = _rng_sub.choice(len(yg), size=MAX_MCMC_ROWS, replace=False)
        Xg = Xg[_idx_sub]; yg = yg[_idx_sub]
    print(f"  Group {gi} ({len(yg)} rows) ...", end=" ", flush=True)
    _t0 = _time.perf_counter()
    with pm.Model():
        baseline = pm.Normal("baseline", mu=0.0, sigma=np.sqrt(SIGMA_SQ_MCMC), shape=n_base)
        z_coup   = pm.NormalMixture(
            "z_coup",
            w=np.array([1.0 - pi_coup, pi_coup]),
            mu=np.zeros(2),
            sigma=np.array([np.sqrt(V0_MCMC), np.sqrt(V1_MCMC)]),
            shape=_G,
        )
        theta   = pm.math.concatenate([baseline, z_coup])
        logit_p = pm.math.dot(Xg, theta)
        _       = pm.Bernoulli("y_obs", logit_p=logit_p, observed=yg.astype(int))
        trace   = pm.sample(N_MCMC_DRAWS, tune=N_MCMC_TUNE, chains=N_MCMC_CHAINS,
                            random_seed=MCMC_SEED + gi, progressbar=False,
                            target_accept=0.85)
    mcmc_traces[gi] = trace
    print(f"done ({_time.perf_counter()-_t0:.1f}s)")

# ── Stack draws ──────────────────────────────────────────────────────────
n_total_draws = N_MCMC_DRAWS * N_MCMC_CHAINS
mcmc_draws    = {}
map_params    = {gi: dev_emvs_result["group_params"][gi] for gi in range(_G)}

for gi in range(_G):
    if mcmc_traces[gi] is None:
        mcmc_draws[gi] = np.tile(map_params[gi], (n_total_draws, 1))
        continue
    tr    = mcmc_traces[gi]
    _base = tr.posterior["baseline"].values.reshape(-1, n_base)
    _coup = tr.posterior["z_coup"].values.reshape(-1, _G)
    mcmc_draws[gi] = np.concatenate([_base, _coup], axis=1)

print(f"\nMCMC complete. Total draws per group: {n_total_draws}")

## Posterior distributions of the scalar parameters

Histograms of the draws across communities; red dashed line = EMVS MAP, shaded band = 95% interval.

In [ ]:
# ── Scalar parameter posteriors ──────────────────────────────────────────
# Parameters: intercept, persistence, direct treatment, spillover
SCALAR_LABELS = ["Intercept", "Persistence", "Direct treatment", "Spillover"]
_SHORT_BINS   = [f"Comm {b}" for b in range(_G)]
_gcols        = [plt.cm.tab10(i) for i in range(_G)]

fig_sc, axes_sc = plt.subplots(_G, 4, figsize=(14, 2.8 * _G), constrained_layout=True)
if _G == 1:
    axes_sc = axes_sc[np.newaxis, :]

for gi in range(_G):
    draws = mcmc_draws[gi]          # (n_total_draws, 4+G)
    map_g = map_params[gi]
    for pi_idx in range(4):
        ax = axes_sc[gi, pi_idx]
        d  = draws[:, pi_idx]
        lo, hi = np.percentile(d, [2.5, 97.5])
        ax.axvspan(lo, hi, alpha=0.18, color=_gcols[gi])
        ax.hist(d, bins=40, color=_gcols[gi], alpha=0.75, density=True, edgecolor="none")
        ax.axvline(map_g[pi_idx], color="#c0392b", lw=1.8, ls="--")
        ax.axvline(0,             color="gray",    lw=0.8, ls=":",  alpha=0.6)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        if gi == 0:
            ax.set_title(SCALAR_LABELS[pi_idx], fontsize=11, fontweight="bold")
        if pi_idx == 0:
            ax.set_ylabel(_SHORT_BINS[gi], fontsize=9, fontweight="bold", color=_gcols[gi])

legend_handles = [
    _mlines.Line2D([0], [0], color="#c0392b", lw=1.8, ls="--", label="EMVS MAP"),
    _mpatches.Patch(fc="steelblue", alpha=0.35, label="95% posterior interval"),
]
fig_sc.legend(handles=legend_handles, loc="lower center", ncol=2, fontsize=9,
              bbox_to_anchor=(0.5, -0.02))
fig_sc.suptitle(
    f"Posterior distributions"
    f"(village {DEV_VILLAGE}, {n_total_draws} draws)",
    fontsize=11, fontweight="bold",
)
savefig(fig_sc, f"posterior_scalar_v{DEV_VILLAGE}")
plt.show()

In [ ]:
# ── Coupling posteriors (G × G grid) ─────────────────────────────────────
fig_cp, axes_cp = plt.subplots(_G, _G, figsize=(3.2 * _G, 2.5 * _G), constrained_layout=True)
if _G == 1:
    axes_cp = np.array([[axes_cp]])
elif axes_cp.ndim == 1:
    axes_cp = axes_cp[np.newaxis, :]

for gi in range(_G):
    draws = mcmc_draws[gi]
    map_g = map_params[gi]
    for mi in range(_G):
        ax = axes_cp[gi, mi]
        d  = draws[:, 4 + mi]
        lo, hi = np.percentile(d, [2.5, 97.5])
        ax.axvspan(lo, hi, alpha=0.18, color=_gcols[gi])
        ax.hist(d, bins=40, color=_gcols[gi], alpha=0.75, density=True, edgecolor="none")
        ax.axvline(map_g[4 + mi], color="#c0392b", lw=1.8, ls="--")
        ax.axvline(0,             color="gray",    lw=0.8, ls=":",  alpha=0.6)
        ax.tick_params(labelsize=6)
        ax.grid(alpha=0.2)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        if gi == 0:
            ax.set_title(f"Neighbor: {_SHORT_BINS[mi]}", fontsize=8, fontweight="bold")
        if mi == 0:
            ax.set_ylabel(f"Focal: {_SHORT_BINS[gi]}", fontsize=8, fontweight="bold",
                          color=_gcols[gi])
        ax.set_xlabel(f"$\\gamma_{{{gi},{mi}}}$", fontsize=8)

fig_cp.suptitle(
    f"Posterior distributions — couplings $\\gamma_{{k,m}}$  "
    f"(village {DEV_VILLAGE}, {n_total_draws} draws)",
    fontsize=11, fontweight="bold",
)
savefig(fig_cp, f"posterior_couplings_v{DEV_VILLAGE}")
plt.show()

# ── Coupling heatmaps: 2×2 grid ───────────────────────────────────────────
# EMVS posterior inclusion probabilities (from EM E-step)
emvs_pip = dev_emvs_result["inclusion_probs"]  # (G, G)

# MCMC inclusion probabilities: fraction of draws with |γ| > spike/slab crossover
_pip_thresh = np.sqrt(
    2 * V0_MCMC * V1_MCMC * np.log(V1_MCMC / V0_MCMC) / (V1_MCMC - V0_MCMC)
)
mcmc_pip = np.array([
    [(np.abs(mcmc_draws[gi][:, 4 + mi]) > _pip_thresh).mean() for mi in range(_G)]
    for gi in range(_G)
])

# EMVS MAP couplings and MCMC posterior std — independent scales
cm_map  = ext_e["coupling_matrix_mean"]
std_mat = np.array([
    [mcmc_draws[gi][:, 4 + mi].std() for mi in range(_G)]
    for gi in range(_G)
])

# Cap MAP colormap at 95th percentile of |values| to avoid outlier saturation
_map_vmax = float(np.percentile(np.abs(cm_map), 95)) if np.abs(cm_map).max() > 0 else 1.0
_std_vmax = float(std_mat.max()) if std_mat.max() > 0 else 1.0

fig_hm, axes_hm = plt.subplots(2, 2, figsize=(4.5 * _G, 4.2 * 2), constrained_layout=True)

heatmap_specs = [
    (axes_hm[0, 0], emvs_pip,
     r"EMVS Inclusion Prob  $\hat{z}_{k,m}$",
     "Blues", 0.0, 1.0),
    (axes_hm[0, 1], mcmc_pip,
     r"MCMC Inclusion Prob  $P(|\gamma|>\tau)$",
     "Blues", 0.0, 1.0),
    (axes_hm[1, 0], cm_map,
     r"EMVS MAP  $\hat{\gamma}_{k,m}$",
     "RdBu_r", -_map_vmax, _map_vmax),
    (axes_hm[1, 1], std_mat,
     r"Posterior Std  $\sigma(\gamma_{k,m})$",
     "Oranges", 0.0, _std_vmax),
]

for ax_h, mat, title, cmap_, vmin, vmax in heatmap_specs:
    im = ax_h.imshow(mat, cmap=cmap_, vmin=vmin, vmax=vmax, aspect="auto")
    ax_h.set_title(title, fontweight="bold", fontsize=10)
    ax_h.set_xticks(range(_G))
    ax_h.set_xticklabels(_SHORT_BINS, rotation=20, ha="right", fontsize=7)
    ax_h.set_yticks(range(_G))
    ax_h.set_yticklabels(_SHORT_BINS, fontsize=7)
    ax_h.set_xlabel("Neighbor community m", fontsize=9)
    ax_h.set_ylabel("Focal community k",    fontsize=9)
    plt.colorbar(im, ax=ax_h, shrink=0.85)
    for i in range(_G):
        for j in range(_G):
            txt = mat[i, j]
            mid = (vmin + vmax) / 2
            light_bg = abs(txt - mid) < 0.4 * (vmax - vmin)
            ax_h.text(j, i, f"{txt:.2f}", ha="center", va="center", fontsize=7,
                      color="black" if light_bg else "white")

fig_hm.suptitle(
    f"Coupling heatmaps — village {DEV_VILLAGE}",
    fontsize=12, fontweight="bold",
)
savefig(fig_hm, f"posterior_couplings_heatmap_v{DEV_VILLAGE}")
plt.show()

## Ensemble policy

N_ENSEMBLE CQL agents, each on a state built from a different posterior draw; actions by majority vote.

In [ ]:
# ── Train ensemble policy from pre-computed MCMC draws ───────────────
# dev_emvs_result and mcmc_draws are already in memory from cells 5 and 7.
# QIsingPolicy.fit() skips EMVS and MCMC when pre-computed results are passed.

_qi_kwargs = dict(
    v0=V0_E, v1=V1_E, c=C_E, sigma_sq=SIGMA_E,
    discount=DISCOUNT, alpha_cql=ALPHA_CQL,
    v0_mcmc=V0_MCMC, v1_mcmc=V1_MCMC, c_prior=C_PRIOR, sigma_sq_mcmc=SIGMA_SQ_MCMC,
    n_mcmc_draws=N_MCMC_DRAWS, n_mcmc_tune=N_MCMC_TUNE, n_mcmc_chains=N_MCMC_CHAINS,
    mcmc_seed=MCMC_SEED, max_mcmc_rows=MAX_MCMC_ROWS,
    n_ensemble=N_ENSEMBLE, ensemble_seed=ENSEMBLE_SEED,
)

print(f"Training {N_ENSEMBLE} ensemble CQL agents from MCMC posterior draws ...")
_t0_ens = _time.perf_counter()
qi_ens_pol = QIsingPolicy(
    ensemble=True, ising_augmentation=True, label="qi_ens", verbose=True, **_qi_kwargs
).fit(
    dev_trajs, dvd["adj"], dvd["block_labels"],
    _emvs_result=dev_emvs_result,
    _mcmc_draws=mcmc_draws,
)
print(f"Ensemble training complete ({_time.perf_counter()-_t0_ens:.0f}s total).")

# Expose for backward-compatible decision-tree cell
ensemble_models = qi_ens_pol._ensemble_models

In [ ]:
# ── Evaluate ensemble and baselines ──────────────────────────────────
_norm = T_TEST * dvd["N"]
_all_b = np.arange(_nb, dtype=np.int32)

def _make_te(seed):
    return EdgeBetCommunitySISEnv(
        dvd["adj"], dvd["beta_vec"], dvd["delta_vec"], dvd["block_labels"],
        T=T_TEST, seed=seed, init_rate=TEST_INIT_RATE)

_ref_env = _make_te(0)

In [ ]:
# ── Ensemble decision tree: vote distribution at each timestep ───────────
VOTE_THRESHOLD = 0
TREE_EPS       = T_TEST

_te_tree = _make_te(seed=2026)
obs_tr   = _te_tree.reset(); done_tr = False; step_tr = 0
tree_records = []

while not done_tr and step_tr < TREE_EPS:
    avl_tr   = set(_te_tree.get_available_actions())
    votes_tr = np.zeros(_nb, dtype=int)
    for _, draw_params, mdl_draw, so_draw in ensemble_models:
        ob = so_draw(obs_tr["adopted"]).reshape(1, -1)
        qv = mdl_draw.predict_value(np.repeat(ob, _nb, axis=0), _all_b)
        for b in np.argsort(qv)[::-1]:
            if int(b) in avl_tr:
                votes_tr[int(b)] += 1; break
    maj_tr = int(np.argmax(votes_tr))
    tree_records.append({
        "t":            step_tr,
        "majority":     maj_tr,
        "vote_counts":  votes_tr.copy(),
        "alternatives": [b for b in range(_nb)
                         if b != maj_tr and votes_tr[b] > VOTE_THRESHOLD],
        "adoption_rate": float(_te_tree.adopted.mean()),
    })
    obs_tr, _, done_tr, _ = _te_tree.step(maj_tr)
    step_tr += 1

# ── Draw the decision tree ────────────────────────────────────────────────
_gcols_t  = [plt.cm.tab10(i) for i in range(_nb)]
fig_t, ax_t = plt.subplots(figsize=(max(13, TREE_EPS * 1.1), 5 + _nb * 0.4))
ax_t.set_xlim(-0.5, TREE_EPS - 0.5)
ax_t.set_ylim(-1, _nb + 0.5)
ax_t.set_xlabel("Time step", fontsize=11)
ax_t.set_ylabel("Community", fontsize=11)
ax_t.set_yticks(range(_nb))
ax_t.set_yticklabels(
    [f"[{b}] {_dev_env_ref.bin_names[b]}" for b in range(_nb)], fontsize=8)
ax_t.set_title(
    f"Ensemble decision tree — majority-vote path  "
    f"(T={TREE_EPS}, {N_ENSEMBLE} agents, branch threshold > {VOTE_THRESHOLD} votes)\n"
    f"Village {DEV_VILLAGE}",
    fontweight="bold", fontsize=10)
ax_t.grid(alpha=0.2, axis="both")
ax_t.set_facecolor("#f8f8f8")

prev_maj = None
for rec in tree_records:
    t, maj, vc = rec["t"], rec["majority"], rec["vote_counts"]
    r = max(0.15, vc[maj] / N_ENSEMBLE * 0.42)
    ax_t.add_patch(plt.Circle((t, maj), r, color="#d62728", zorder=3, alpha=0.85))
    ax_t.text(t, maj, str(vc[maj]), ha="center", va="center",
              fontsize=8, fontweight="bold", color="white", zorder=4)
    if prev_maj is not None:
        ax_t.plot([t-1, t], [prev_maj, maj], "-", color="#d62728", lw=2.0, zorder=2, alpha=0.7)
    for alt in rec["alternatives"]:
        r_a = max(0.10, vc[alt] / N_ENSEMBLE * 0.32)
        ax_t.add_patch(plt.Circle((t + 0.38, alt), r_a, color="#ff7f0e", zorder=3, alpha=0.6))
        ax_t.plot([t, t + 0.38], [maj, alt], ":", color="#ff7f0e", lw=1.2, zorder=2)
        ax_t.text(t + 0.38, alt, str(vc[alt]), ha="center", va="center",
                  fontsize=7, color="white", zorder=4)
    prev_maj = maj

ax_t.legend(handles=[
    _mpatches.Patch(color="#d62728", label="Majority vote (main path)"),
    _mpatches.Patch(color="#ff7f0e", label=f"Alternative (> {VOTE_THRESHOLD} votes)"),
], loc="upper right", fontsize=9)

ax2_t = ax_t.twinx()
ax2_t.plot([r["t"] for r in tree_records],
           [r["adoption_rate"] for r in tree_records],
           "k--", lw=1.3, alpha=0.5, label="Adoption rate")
ax2_t.set_ylabel("Adoption rate", fontsize=9, color="grey")
ax2_t.tick_params(axis="y", labelcolor="grey")
ax2_t.set_ylim(0, 1)

plt.tight_layout()
savefig(fig_t, f"ensemble_decision_tree_v{DEV_VILLAGE}")
plt.show()